# Generate Enhanced Query pkl file


## Setup

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21 (required for Pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
# Note: Using faiss-cpu (faiss-gpu has compatibility issues)
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch
!pip install -q datasets  # For HuggingFace data loading
!pip install -q accelerate bitsandbytes  # For Qwen 2.5 3B

print("\n" + "="*60)
print("✓ Installation complete")
print("="*60)
print("⚠️ IMPORTANT: Restart runtime now!")
print("   1. Click 'Runtime' → 'Restart runtime'")
print("   2. Then run cells starting from 'Step 2' below")
print("="*60)

Cloning into 'graduation'...
remote: Enumerating objects: 524, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 524 (delta 16), reused 51 (delta 14), pack-reused 468 (from 1)
Receiving objects: 100% (524/524), 20.10 MiB | 17.89 MiB/s, done.
Resolving deltas: 100% (188/188), done.
/content/graduation/arabic-rag-query-enhancement
Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /usr/lib/

### Step 2: Mount Google Drive and Configure Environment (Run After Restart)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project
%cd /content/graduation/arabic-rag-query-enhancement

# Configure environment
import os
import sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Add src to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

print("\n✓ Environment configured")
print("✓ Ready to run experiment")

Mounted at /content/drive
/content/graduation/arabic-rag-query-enhancement
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)

✓ Environment configured
✓ Ready to run experiment


### Step 3: Setup Symbolic Links to Google Drive Index

In [ ]:
# Create data directory
!mkdir -p data/miracl_ar

Verifying index files...
Index exists: False
Docid map exists: False

⚠️ ERROR: Index not found!
Please update 'drive_base' path above


## Import Modules

In [ ]:
from src.utils.data_loader import MIRACLDataLoader
from src.retrievers.dense import mDPRRetriever
from src.enhancers.query2doc import Query2DocEnhancer
from src.evaluation.metrics import RetrievalEvaluator, save_results, save_metrics, print_metrics

import torch
from tqdm.notebook import tqdm

print("✓ Modules imported")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

✓ Modules imported
GPU Available: True
GPU: Tesla T4


## Load Data

In [ ]:
# Load MIRACL Arabic dev set
data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

print(f"\nDataset Statistics:")
print(f"  Queries: {len(topics)}")
print(f"  Qrels: {len(qrels)}")

# Show sample
sample_qid = list(topics.keys())[0]
print(f"\nSample Query:")
print(f"  ID: {sample_qid}")
print(f"  Text: {topics[sample_qid]['title']}")
print(f"  Relevant docs: {len(qrels.get(sample_qid, {}))}")

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries

Dataset Statistics:
  Queries: 2896
  Qrels: 2896

Sample Query:
  ID: 8099
  Text: من هو علي بن محمد السمري؟
  Relevant docs: 10


## Initialize Components

In [ ]:
# Initialize Query2Doc enhancer
print("Initializing Query2Doc enhancer...")
print("This will download Qwen 2.5 3B (~6GB) on first run")

import importlib
import sys
if 'src.enhancers.query2doc' in sys.modules:
    del sys.modules['src.enhancers.query2doc']
from src.enhancers.query2doc import Query2DocEnhancer



enhancer = Query2DocEnhancer(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_new_tokens=128,  # Pseudo-document length
    temperature=0.7,     # Low temperature for focused generation
    top_p=0.9,
    batch_size=8
)

print("✓ Query2Doc enhancer ready")

Initializing Query2Doc enhancer...
This will download Qwen 2.5 3B (~6GB) on first run
Loading Qwen/Qwen2.5-3B-Instruct in float16...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Model loaded on cuda:0
✓ Batch size: 8 (processing 8 queries at once)
✓ Max tokens: 128 (shorter = faster)
✓ Query2Doc enhancer ready


In [ ]:
# Test enhancer on sample query
sample_query = topics[sample_qid]['title']
print(f"Testing enhancer on sample query...\n")
print(f"Original: {sample_query}")
print(f"\nGenerating pseudo-document...")

enhanced_sample = enhancer.enhance(sample_query)
print(f"\nEnhanced: {enhanced_sample[:500]}...")  # Show first 500 chars
print(f"\nLength: {len(enhanced_sample)} chars")

Testing enhancer on sample query...

Original: من هو علي بن محمد السمري؟

Generating pseudo-document...

Enhanced: من هو علي بن محمد السمري؟ علي بن محمد السمري هو شخصية تاريخية عربية من القرن السابع الميلادي، وهو أحد العلماء والشاعر الفاطميين. كان معلماً جامعياً في مصر وله تأثير كبير في حقل الفلسفة الإسلامية. شغل منصب قاضٍ ووزير في الدولة الفاطمية. ساهم في نشر الفكر العربي الإسلامي في مصر وساهم أيضاً في ترجمة الكتب اليونانية والفارسية إلى اللغة العربية....

Length: 342 chars


## Run Experiment

In [ ]:
print("="*60)
print("EXPERIMENT 003: Query2Doc + Dense Retrieval")
print("="*60)

# Prepare queries
query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nTotal queries: {len(query_texts)}")

EXPERIMENT 003: Query2Doc + Dense Retrieval

Total queries: 2896


In [ ]:
# Apply Query2Doc enhancement
print(f"\nApplying Query2Doc enhancement...")
print("This will take ~15-20 minutes for all queries")
print("(~2-3 seconds per query for generation)\n")

enhanced_queries = enhancer.enhance_batch(
    query_texts,
    query_ids,
    show_progress=True
)

print(f"\n✓ Enhanced {len(enhanced_queries)} queries")


Applying Query2Doc enhancement...
This will take ~15-20 minutes for all queries
(~2-3 seconds per query for generation)



Enhancing batches: 100%|██████████| 362/362 [41:49<00:00,  6.93s/it]


✓ Enhanced 2896 queries


In [ ]:
# Show enhancement examples
print("\nEnhancement Examples:\n")
for i in range(min(3, len(query_texts))):
    print(f"Query {i+1}:")
    print(f"  Original: {query_texts[i]}")
    print(f"  Enhanced: {enhanced_queries[i][:200]}...")  # First 200 chars
    print()


Enhancement Examples:

Query 1:
  Original: من هو علي بن محمد السمري؟
  Enhanced: من هو علي بن محمد السمري؟ علي بن محمد السمري هو شخصية إسلامية معروفة في التاريخ الإسلامي، وُلد حوالي عام 132 هجري (652 ميلادي). كان عالماً ومحدثاً ومتكلماً، وهو من أهل المدينة المنورة. كتب في فقه العق...

Query 2:
  Original: متى تم إستخدام الغوّاصات لأول مرة؟
  Enhanced: متى تم إستخدام الغوّاصات لأول مرة؟ تم استخدام الغواصات لأول مرة في الحرب العالمية الأولى (1914-1918). حيث قام الألمان بإنشاء أول غواصة فعالة في العالم وهي "السفن السفينة" (U-boat) في عام 1906. ومع ذلك...

Query 3:
  Original: من هو القديس المسمى بالصخرة؟
  Enhanced: من هو القديس المسمى بالصخرة؟ القديس المسمى بالصخرة هو القديس بطرس، وهو واحد من أتباع السيد المسيح وأول أساقفة روما. ورد في الإنجيل حديث يشير إلى أن السيد المسيح أشار إليه بالصخرة عندما قال: "أنت الصخر...



In [ ]:
# Save enhanced queries
import pickle
with open('enhanced_queries_exp003.pkl', 'wb') as f:
    pickle.dump({'query_ids': query_ids, 'original': query_texts, 'enhanced': enhanced_queries}, f)
print("✓ Enhanced queries saved to: enhanced_queries_exp003.pkl")

✓ Enhanced queries saved to: enhanced_queries_exp003.pkl
